In [ ]:
# Upload datasets as .csv files in your colab session, and then run this cell.
# Dataset Link: https://drive.google.com/drive/folders/1PzURxyRaN5rclRcyRSI9G8oa1XVSo_SF?usp=drive_link

# Don't work in this colab. Make a copy in your drive instead.

import pandas as pd

copd_df = '/content/COPD_data_preprocessed.csv'
alzheimers_df = '/content/alzheimers_disease_data_preprocessed.csv'
arrhythmia_df = '/content/arrhythmia_data_preprocessed.csv'
autism_df = '/content/autism_screening_data_preprocessed.csv'
diabetes_df = '/content/diabetes_data_preprocessed.csv'
stroke_df = '/content/healthcare-dataset-stroke-data_preprocessed.csv'
heart_df = '/content/heart_disease_uci_preprocessed.csv'
hepatitis_df = '/content/hepatitis_data_preprocessed.csv'
liver_df = '/content/liver_patient_dataset_preprocessed.csv'
thyroid_df = '/content/thyroid_dataset_preprocessed.csv'

In [ ]:
# Cell 2: Install Bayesian Optimization (Optuna) & dependencies
!pip install -q optuna xgboost statsmodels scikit-learn

In [ ]:
# Cell 3: Configuration & Dataset Registry
import gc
import warnings
import numpy as np
import pandas as pd
import sklearn
import statsmodels.api as sm
from statsmodels.tools.sm_exceptions import ConvergenceWarning, PerfectSeparationWarning
from tqdm.auto import tqdm

import optuna
from xgboost import XGBClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import KNNImputer
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.metrics import roc_auc_score, f1_score, accuracy_score
from sklearn.exceptions import ConvergenceWarning as SklearnConvergenceWarning

# Silence Optuna per-trial spam; keep progress clean
optuna.logging.set_verbosity(optuna.logging.WARNING)

# Check sklearn version for LogisticRegression API compatibility (1.8+ deprecated `penalty`)
SKLEARN_VERSION = tuple(int(x) for x in sklearn.__version__.split(".")[:2])
print(f"scikit-learn version: {sklearn.__version__}")

# ==========================================
# Global Benchmark Configuration
# ==========================================
RANDOM_STATE = 42       # Identical outer CV partitions across all models
N_SPLITS = 5            # Outer 5-fold stratified CV
INNER_SPLITS = 3        # Inner 3-fold stratified CV for Bayesian Optimization
N_OPTUNA_TRIALS = 5    # Bayesian Optimization trials per outer fold
THRESHOLD = 0.5         # Fixed classification threshold
KNN_NEIGHBORS = 5       # KNN imputation neighbors
NOISE_FRACTION = 0.10   # 10% of training-set feature SD for robustness noise

DATASETS = [
    ("COPD", copd_df, "Pulmonology / Respiratory"),
    ("Alzheimers", alzheimers_df, "Neurology"),
    ("Arrhythmia", arrhythmia_df, "Cardiology"),
    ("Autism", autism_df, "Neurodevelopmental / Psychiatry"),
    ("Diabetes", diabetes_df, "Endocrinology"),
    ("Stroke", stroke_df, "Neurology / Cerebrovascular"),
    ("Heart_Disease", heart_df, "Cardiology"),
    ("Hepatitis", hepatitis_df, "Hepatology / Infectious Disease"),
    ("Liver_Disease", liver_df, "Hepatology"),
    ("Thyroid", thyroid_df, "Endocrinology"),
]

scikit-learn version: 1.6.1


In [ ]:
# Cell 4: Exact Metric Implementations (Protocol Sections 3 & 4)

def _safe_div(num, den):
    return float(num / den) if den > 0 else np.nan


def compute_fairness_metrics(y_true, y_pred, sex):
    y_true = np.asarray(y_true, dtype=int)
    y_pred = np.asarray(y_pred, dtype=int)
    sex = np.asarray(sex, dtype=int)

    stats = {}
    for group_name, group_val in [("Male", 1), ("Female", 0)]:
        mask = (sex == group_val)
        if np.sum(mask) == 0:
            stats[group_name] = {"SR": np.nan, "TPR": np.nan, "FPR": np.nan, "PPV": np.nan}
            continue

        yt = y_true[mask]
        yp = y_pred[mask]

        tp = np.sum((yt == 1) & (yp == 1))
        tn = np.sum((yt == 0) & (yp == 0))
        fp = np.sum((yt == 0) & (yp == 1))
        fn = np.sum((yt == 1) & (yp == 0))

        stats[group_name] = {
            "SR": float(np.mean(yp == 1)),
            "TPR": _safe_div(tp, tp + fn),
            "FPR": _safe_div(fp, fp + tn),
            "PPV": _safe_div(tp, tp + fp),
        }

    sr_m, sr_f = stats["Male"]["SR"], stats["Female"]["SR"]
    tpr_m, tpr_f = stats["Male"]["TPR"], stats["Female"]["TPR"]
    fpr_m, fpr_f = stats["Male"]["FPR"], stats["Female"]["FPR"]
    ppv_m, ppv_f = stats["Male"]["PPV"], stats["Female"]["PPV"]

    di = np.nan if (np.isnan(sr_m) or np.isnan(sr_f) or max(sr_m, sr_f) == 0.0) else float(min(sr_m, sr_f) / max(sr_m, sr_f))
    eod = np.nan if (np.isnan(tpr_m) or np.isnan(tpr_f)) else float(abs(tpr_m - tpr_f))
    dpd = np.nan if (np.isnan(sr_m) or np.isnan(sr_f)) else float(abs(sr_m - sr_f))
    eqod = np.nan if any(np.isnan(v) for v in [tpr_m, tpr_f, fpr_m, fpr_f]) else float(max(abs(tpr_m - tpr_f), abs(fpr_m - fpr_f)))
    ppd = np.nan if (np.isnan(ppv_m) or np.isnan(ppv_f)) else float(abs(ppv_m - ppv_f))

    return {"DI": di, "EOD": eod, "DPD": dpd, "EqOD": eqod, "PPD": ppd}


def compute_discrimination_metrics(y_true, y_pred, y_prob):
    y_true = np.asarray(y_true, dtype=int)
    y_pred = np.asarray(y_pred, dtype=int)
    y_prob = np.asarray(y_prob, dtype=np.float64)

    tp = np.sum((y_true == 1) & (y_pred == 1))
    tn = np.sum((y_true == 0) & (y_pred == 0))
    fp = np.sum((y_true == 0) & (y_pred == 1))
    fn = np.sum((y_true == 1) & (y_pred == 0))

    return {
        "AUROC": float(roc_auc_score(y_true, y_prob)) if len(np.unique(y_true)) == 2 else np.nan,
        "Sensitivity": _safe_div(tp, tp + fn),
        "Specificity": _safe_div(tn, tn + fp),
        "PPV": _safe_div(tp, tp + fp),
        "NPV": _safe_div(tn, tn + fn),
    }


def compute_calibration_metrics(y_true, y_prob):
    y_true = np.asarray(y_true, dtype=np.float64)
    p_raw = np.asarray(y_prob, dtype=np.float64)
    p_clip = np.clip(p_raw, 1e-15, 1.0 - 1e-15)

    # 1. ECE: 15 equal-width bins covering [0, 1]
    bin_edges = np.linspace(0.0, 1.0, 16)
    n_total = len(y_true)
    ece = 0.0
    for b in range(15):
        mask = (p_clip >= bin_edges[b]) & (p_clip < bin_edges[b + 1]) if b < 14 else (p_clip >= bin_edges[b]) & (p_clip <= bin_edges[b + 1])
        n_b = np.sum(mask)
        if n_b > 0:
            ece += (n_b / n_total) * abs(np.mean(y_true[mask]) - np.mean(p_clip[mask]))

    # 2. NLL
    nll = -np.mean(y_true * np.log(p_clip) + (1.0 - y_true) * np.log(1.0 - p_clip))

    # 3. Brier Score
    brier = np.mean((p_raw - y_true) ** 2)

    # 4-5. Calibration Slope & Intercept (catch non-convergence & perfect separation as N/A)
    cal_intercept = np.nan
    cal_slope = np.nan
    logit_p = np.log(p_clip / (1.0 - p_clip))

    if len(np.unique(y_true)) == 2 and np.std(logit_p) > 0:
        try:
            with warnings.catch_warnings():
                warnings.simplefilter("error", ConvergenceWarning)
                warnings.simplefilter("error", PerfectSeparationWarning)
                X_design = sm.add_constant(logit_p, has_constant="add")
                fit_res = sm.Logit(y_true, X_design).fit(disp=False, maxiter=1000)
                if fit_res.mle_retvals.get("converged", False):
                    cal_intercept = float(fit_res.params[0])
                    cal_slope = float(fit_res.params[1])
        except Exception:
            cal_intercept = np.nan
            cal_slope = np.nan

    return {
        "ECE": float(ece),
        "NLL": float(nll),
        "Brier": float(brier),
        "Cal_Slope": cal_slope,
        "Cal_Intercept": cal_intercept,
    }


def compute_robustness_metrics(y_true, y_pred_clean, p_clean, y_pred_noisy, p_noisy):
    y_true = np.asarray(y_true, dtype=int)
    y_pred_clean = np.asarray(y_pred_clean, dtype=int)
    y_pred_noisy = np.asarray(y_pred_noisy, dtype=int)
    p_clean = np.asarray(p_clean, dtype=np.float64)
    p_noisy = np.asarray(p_noisy, dtype=np.float64)

    if len(np.unique(y_true)) == 2:
        auroc_deg = float(roc_auc_score(y_true, p_clean) - roc_auc_score(y_true, p_noisy))
    else:
        auroc_deg = np.nan

    acc_deg = float(accuracy_score(y_true, y_pred_clean) - accuracy_score(y_true, y_pred_noisy))
    f1_deg = float(
        f1_score(y_true, y_pred_clean, pos_label=1, zero_division=0.0)
        - f1_score(y_true, y_pred_noisy, pos_label=1, zero_division=0.0)
    )
    flip_rate = float(np.mean(y_pred_clean != y_pred_noisy))
    mapc = float(np.mean(np.abs(p_clean - p_noisy)))

    return {
        "AUROC_Deg": auroc_deg,
        "Accuracy_Deg": acc_deg,
        "F1_Deg": f1_deg,
        "Flip_Rate": flip_rate,
        "MAPC": mapc,
    }


def summarize_fold_values(values):
    arr = np.asarray(values, dtype=np.float64)
    valid = arr[~np.isnan(arr)]
    if len(valid) == 0:
        return np.nan, np.nan
    mean_val = float(np.mean(valid))
    sd_val = float(np.std(valid, ddof=1)) if len(valid) >= 2 else np.nan
    return mean_val, sd_val

In [ ]:
# Cell 5: Bayesian Optimization Tuning & Model Builders (Zero Data Leakage)

def _make_lr(C, l1_ratio):
    kwargs = dict(
        C=C,
        l1_ratio=l1_ratio,
        solver="saga",a
        max_iter=3000,
        tol=1e-4,
        random_state=RANDOM_STATE,
    )
    if SKLEARN_VERSION < (1, 8):
        kwargs["penalty"] = "elasticnet"
    return LogisticRegression(**kwargs)


def build_lr_model(params):
    return Pipeline([
        ("scaler", StandardScaler()),
        ("model", _make_lr(C=params["C"], l1_ratio=params["l1_ratio"])),
    ])


def tune_lr_bayesian(X_train_raw, y_train, seed):
    inner_cv = StratifiedKFold(n_splits=INNER_SPLITS, shuffle=True, random_state=seed)

    def objective(trial):
        C = trial.suggest_float("C", 1e-4, 1e2, log=True)
        l1_ratio = trial.suggest_float("l1_ratio", 0.0, 1.0)

        pipe = Pipeline([
            ("imputer", KNNImputer(n_neighbors=KNN_NEIGHBORS)),
            ("scaler", StandardScaler()),
            ("model", _make_lr(C=C, l1_ratio=l1_ratio)),
        ])
        with warnings.catch_warnings():
            warnings.simplefilter("ignore", SklearnConvergenceWarning)
            scores = cross_val_score(pipe, X_train_raw, y_train, cv=inner_cv, scoring="roc_auc", n_jobs=-1)
        return float(np.mean(scores))

    study = optuna.create_study(
        direction="maximize",
        sampler=optuna.samplers.TPESampler(seed=seed),
    )
    study.optimize(objective, n_trials=N_OPTUNA_TRIALS, show_progress_bar=False)
    return study.best_params


def build_xgb_model(params):
    return XGBClassifier(
        **params,
        eval_metric="logloss",
        random_state=RANDOM_STATE,
        n_jobs=-1,
    )


def tune_xgb_bayesian(X_train_raw, y_train, seed):
    inner_cv = StratifiedKFold(n_splits=INNER_SPLITS, shuffle=True, random_state=seed)

    def objective(trial):
        params = {
            "n_estimators": trial.suggest_int("n_estimators", 50, 400),
            "max_depth": trial.suggest_int("max_depth", 2, 8),
            "learning_rate": trial.suggest_float("learning_rate", 1e-2, 0.3, log=True),
            "subsample": trial.suggest_float("subsample", 0.6, 1.0),
            "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 1.0),
            "min_child_weight": trial.suggest_int("min_child_weight", 1, 10),
            "reg_alpha": trial.suggest_float("reg_alpha", 1e-8, 10.0, log=True),
            "reg_lambda": trial.suggest_float("reg_lambda", 1e-8, 10.0, log=True),
        }
        pipe = Pipeline([
            ("imputer", KNNImputer(n_neighbors=KNN_NEIGHBORS)),
            ("model", XGBClassifier(**params, eval_metric="logloss", random_state=RANDOM_STATE, n_jobs=1)),
        ])
        scores = cross_val_score(pipe, X_train_raw, y_train, cv=inner_cv, scoring="roc_auc", n_jobs=-1)
        return float(np.mean(scores))

    study = optuna.create_study(
        direction="maximize",
        sampler=optuna.samplers.TPESampler(seed=seed),
    )
    study.optimize(objective, n_trials=N_OPTUNA_TRIALS, show_progress_bar=False)
    return study.best_params

In [ ]:
# Cell 6: Evaluate Logistic Regression and XGBoost, Save Separate CSVs
CSV_COLUMNS = [
    "Dataset", "Train_N", "Test_N", "Num_Feats", "Domain",
    "DI_mean", "DI_sd",
    "EOD_mean", "EOD_sd",
    "DPD_mean", "DPD_sd",
    "EqOD_mean", "EqOD_sd",
    "PPD_mean", "PPD_sd",
    "AUROC_mean", "AUROC_sd",
    "Sensitivity_mean", "Sensitivity_sd",
    "Specificity_mean", "Specificity_sd",
    "PPV_mean", "PPV_sd",
    "NPV_mean", "NPV_sd",
    "ECE_mean", "ECE_sd",
    "NLL_mean", "NLL_sd",
    "Brier_mean", "Brier_sd",
    "Cal_Slope_mean", "Cal_Slope_sd",
    "Cal_Intercept_mean", "Cal_Intercept_sd",
    "AUROC_Deg_mean", "AUROC_Deg_sd",
    "Accuracy_Deg_mean", "Accuracy_Deg_sd",
    "F1_Deg_mean", "F1_Deg_sd",
    "Flip_Rate_mean", "Flip_Rate_sd",
    "MAPC_mean", "MAPC_sd",
]

METRIC_KEYS = [
    "DI", "EOD", "DPD", "EqOD", "PPD",
    "AUROC", "Sensitivity", "Specificity", "PPV", "NPV",
    "ECE", "NLL", "Brier", "Cal_Slope", "Cal_Intercept",
    "AUROC_Deg", "Accuracy_Deg", "F1_Deg", "Flip_Rate", "MAPC",
]


def run_baseline_benchmark(model_name, tune_fn, build_fn, output_csv):
    all_results = []
    skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)
    rng = np.random.default_rng(RANDOM_STATE)

    total_folds = len(DATASETS) * N_SPLITS

    with tqdm(total=total_folds, desc=f"{model_name} Benchmark", unit="fold") as pbar:
        for ds_idx, (ds_name, ds_source, ds_domain) in enumerate(DATASETS):
            df = pd.read_csv(ds_source) if isinstance(ds_source, str) else ds_source.copy()

            sex_col = df.columns[0]
            target_col = df.columns[-1]
            feature_cols = list(df.columns[:-1])

            X = df[feature_cols].copy()
            y = df[target_col].astype(int).copy()

            fold_metrics = {k: [] for k in METRIC_KEYS}
            train_sizes, test_sizes = [], []

            for fold_idx, (train_idx, test_idx) in enumerate(skf.split(X, y), start=1):
                pbar.set_description(f"{model_name} [{ds_name} | Fold {fold_idx}/{N_SPLITS}]")

                X_train_raw = X.iloc[train_idx].copy()
                X_test_raw = X.iloc[test_idx].copy()
                y_train = y.iloc[train_idx].to_numpy(dtype=int)
                y_test = y.iloc[test_idx].to_numpy(dtype=int)

                train_sizes.append(len(train_idx))
                test_sizes.append(len(test_idx))

                # 1. Bayesian Optimization strictly inside outer training fold
                fold_seed = RANDOM_STATE + ds_idx * 100 + fold_idx
                best_params = tune_fn(X_train_raw, y_train, seed=fold_seed)

                # 2. Outer-fold KNN imputation (fit on train, transform train & test)
                imputer = KNNImputer(n_neighbors=KNN_NEIGHBORS)
                X_train_imp = pd.DataFrame(
                    imputer.fit_transform(X_train_raw),
                    columns=feature_cols,
                    index=X_train_raw.index,
                )
                X_test_clean = pd.DataFrame(
                    imputer.transform(X_test_raw),
                    columns=feature_cols,
                    index=X_test_raw.index,
                )

                X_train_imp[sex_col] = np.round(X_train_imp[sex_col]).astype(int)
                X_test_clean[sex_col] = np.round(X_test_clean[sex_col]).astype(int)
                sex_test = X_test_clean[sex_col].to_numpy(dtype=int)

                # 3. Fit tuned model on outer training fold & evaluate clean test fold
                clf = build_fn(best_params)
                with warnings.catch_warnings():
                    warnings.simplefilter("ignore", SklearnConvergenceWarning)
                    clf.fit(X_train_imp, y_train)

                pos_idx = list(clf.classes_).index(1)
                p_clean = np.asarray(clf.predict_proba(X_test_clean)[:, pos_idx], dtype=np.float64)
                y_pred_clean = (p_clean >= THRESHOLD).astype(int)

                # 4. Gaussian noise injection on test features (all except "sex")
                X_test_noisy = X_test_clean.copy()
                for col in feature_cols:
                    if col == sex_col:
                        continue
                    sd_train = float(X_train_imp[col].std(ddof=1))
                    if np.isnan(sd_train) or sd_train == 0.0:
                        continue
                    noise_scale = NOISE_FRACTION * sd_train
                    eps = rng.normal(loc=0.0, scale=1.0, size=len(X_test_noisy))
                    X_test_noisy[col] = X_test_noisy[col] + noise_scale * eps

                p_noisy = np.asarray(clf.predict_proba(X_test_noisy)[:, pos_idx], dtype=np.float64)
                y_pred_noisy = (p_noisy >= THRESHOLD).astype(int)

                # 5. Compute all 20 benchmark metrics
                fold_res = {}
                fold_res.update(compute_fairness_metrics(y_test, y_pred_clean, sex_test))
                fold_res.update(compute_discrimination_metrics(y_test, y_pred_clean, p_clean))
                fold_res.update(compute_calibration_metrics(y_test, p_clean))
                fold_res.update(compute_robustness_metrics(y_test, y_pred_clean, p_clean, y_pred_noisy, p_noisy))

                for k in METRIC_KEYS:
                    fold_metrics[k].append(fold_res[k])

                pbar.set_postfix(
                    AUROC=f"{fold_res['AUROC']:.4f}",
                    ECE=f"{fold_res['ECE']:.4f}",
                    Flip=f"{fold_res['Flip_Rate']:.4f}",
                )
                pbar.update(1)

            avg_train_n = float(np.mean(train_sizes))
            avg_test_n = float(np.mean(test_sizes))
            row = {
                "Dataset": ds_name,
                "Train_N": int(avg_train_n) if avg_train_n.is_integer() else avg_train_n,
                "Test_N": int(avg_test_n) if avg_test_n.is_integer() else avg_test_n,
                "Num_Feats": len(feature_cols),
                "Domain": ds_domain,
            }
            for k in METRIC_KEYS:
                m, s = summarize_fold_values(fold_metrics[k])
                row[f"{k}_mean"] = m
                row[f"{k}_sd"] = s

            all_results.append(row)
            gc.collect()

    results_df = pd.DataFrame(all_results)[CSV_COLUMNS]
    results_df.to_csv(output_csv, index=False, na_rep="N/A")
    print(f"\nSaved {output_csv} with shape {results_df.shape}")
    return results_df


# Run both baselines sequentially
lr_results_df = run_baseline_benchmark(
    model_name="LogisticRegression",
    tune_fn=tune_lr_bayesian,
    build_fn=build_lr_model,
    output_csv="LogisticRegression_Results.csv",
)

xgb_results_df = run_baseline_benchmark(
    model_name="XGBoost",
    tune_fn=tune_xgb_bayesian,
    build_fn=build_xgb_model,
    output_csv="XGBoost_Results.csv",
)

LogisticRegression Benchmark:   0%|          | 0/50 [00:00<?, ?fold/s]

/usr/local/lib/python3.13/dist-packages/statsmodels/discrete/discrete_model.py:2620: RuntimeWarning: overflow encountered in exp
  return 1 / (1 + np.exp(-X))



Saved LogisticRegression_Results.csv with shape (10, 45)


XGBoost Benchmark:   0%|          | 0/50 [00:00<?, ?fold/s]


Saved XGBoost_Results.csv with shape (10, 45)


In [ ]:
# Cell 7: Preview & Download Both Baseline Result Files
import IPython.display as ipydisplay

print("=== Logistic Regression Results ===")
ipydisplay.display(lr_results_df)

print("\n=== XGBoost Results ===")
ipydisplay.display(xgb_results_df)

try:
    from google.colab import files
    files.download("LogisticRegression_Results.csv")
    files.download("XGBoost_Results.csv")
except ImportError:
    pass

=== Logistic Regression Results ===


,Dataset,Train_N,Test_N,Num_Feats,Domain,DI_mean,DI_sd,EOD_mean,EOD_sd,DPD_mean,...,AUROC_Deg_mean,AUROC_Deg_sd,Accuracy_Deg_mean,Accuracy_Deg_sd,F1_Deg_mean,F1_Deg_sd,Flip_Rate_mean,Flip_Rate_sd,MAPC_mean,MAPC_sd
0,COPD,307.2,76.8,14,Pulmonology / Respiratory,0.380782,0.080320,0.315503,0.173400,0.407362,...,0.004876,0.004869,-0.000103,0.022716,-0.000043,0.026436,0.026077,0.013117,0.021591,0.004389
1,Alzheimers,560.0,140.0,8,Neurology,0.924832,0.036391,0.075188,0.055329,0.040616,...,0.000408,0.004145,-0.004286,0.009583,-0.005002,0.010209,0.021429,0.010102,0.020788,0.002786
2,Arrhythmia,331.2,82.8,262,Cardiology,0.611683,0.200216,0.112343,0.109017,0.242307,...,-0.002929,0.005992,0.004761,0.018388,0.003034,0.022678,0.038613,0.015642,0.025543,0.013987
3,Autism,299.2,74.8,19,Neurodevelopmental / Psychiatry,0.863884,0.091825,0.002976,0.006655,0.076254,...,0.000730,0.001633,0.005369,0.007353,0.005189,0.007110,0.010775,0.017613,0.008041,0.002817
4,Diabetes,560.0,140.0,8,Endocrinology,0.852799,0.138565,0.067525,0.051419,0.086098,...,0.001714,0.003618,0.007143,0.011294,0.007685,0.009759,0.021429,0.008748,0.019337,0.001490
5,Stroke,398.4,99.6,10,Neurology / Cerebrovascular,0.844596,0.073766,0.088391,0.067800,0.091949,...,0.000993,0.006031,-0.008020,0.026004,-0.009072,0.024250,0.040182,0.021322,0.013876,0.003817
6,Heart_Disease,560.0,140.0,14,Cardiology,0.254535,0.106076,0.360882,0.176734,0.435240,...,-0.000531,0.002417,0.014286,0.007143,0.013945,0.007567,0.017143,0.006389,0.013297,0.001196
7,Hepatitis,51.2,12.8,19,Hepatology / Infectious Disease,0.000000,0.000000,NaN,NaN,0.497475,...,0.000794,0.018271,0.015385,0.034401,0.029091,0.065049,0.015385,0.034401,0.015794,0.014087
8,Liver_Disease,560.0,140.0,10,Hepatology,0.843133,0.174317,0.094639,0.101904,0.059414,...,0.003143,0.007796,-0.002857,0.015649,-0.005696,0.023049,0.045714,0.018626,0.024597,0.005691
9,Thyroid,560.0,140.0,10,Endocrinology,0.879196,0.139101,0.102762,0.092431,0.048166,...,0.004776,0.016052,-0.001429,0.011737,-0.009293,0.010139,0.044286,0.034773,0.027739,0.016401



=== XGBoost Results ===


,Dataset,Train_N,Test_N,Num_Feats,Domain,DI_mean,DI_sd,EOD_mean,EOD_sd,DPD_mean,...,AUROC_Deg_mean,AUROC_Deg_sd,Accuracy_Deg_mean,Accuracy_Deg_sd,F1_Deg_mean,F1_Deg_sd,Flip_Rate_mean,Flip_Rate_sd,MAPC_mean,MAPC_sd
0,COPD,307.2,76.8,14,Pulmonology / Respiratory,0.413358,0.092329,0.345224,0.138254,0.376870,...,0.060516,0.022663,0.075496,0.016806,0.116722,0.016861,0.184860,0.035810,0.145507,0.014273
1,Alzheimers,560.0,140.0,8,Neurology,0.950153,0.034498,0.073360,0.073999,0.026766,...,0.022551,0.018440,0.025714,0.022361,0.032321,0.023772,0.077143,0.021665,0.059356,0.016143
2,Arrhythmia,331.2,82.8,262,Cardiology,0.548620,0.149491,0.208995,0.095152,0.277791,...,0.008054,0.023298,0.007170,0.039653,0.006032,0.039618,0.065237,0.039637,0.046384,0.016083
3,Autism,299.2,74.8,19,Neurodevelopmental / Psychiatry,0.864218,0.108950,0.000000,0.000000,0.075502,...,0.033866,0.013721,0.074847,0.022234,0.081646,0.026400,0.074847,0.022234,0.073463,0.017236
4,Diabetes,560.0,140.0,8,Endocrinology,0.874430,0.147257,0.090704,0.031676,0.067256,...,0.022551,0.010558,0.020000,0.028302,0.027605,0.031836,0.080000,0.028302,0.052576,0.016159
5,Stroke,398.4,99.6,10,Neurology / Cerebrovascular,0.800422,0.106827,0.177532,0.117872,0.129958,...,0.003308,0.009593,0.004061,0.015229,0.005225,0.017813,0.064263,0.023176,0.037002,0.009749
6,Heart_Disease,560.0,140.0,14,Cardiology,0.322800,0.098959,0.218829,0.185894,0.390317,...,0.024612,0.008616,0.011429,0.017928,0.015511,0.018284,0.062857,0.022246,0.058216,0.017571
7,Hepatitis,51.2,12.8,19,Hepatology / Infectious Disease,0.636364,0.552978,NaN,NaN,0.181818,...,0.038095,0.049372,0.078205,0.143938,0.077855,0.124314,0.108974,0.116272,0.069113,0.050084
8,Liver_Disease,560.0,140.0,10,Hepatology,0.826744,0.122758,0.093863,0.095145,0.085525,...,0.154939,0.017850,0.177143,0.042076,0.157015,0.034314,0.262857,0.042076,0.197228,0.043338
9,Thyroid,560.0,140.0,10,Endocrinology,0.889055,0.107373,0.073526,0.068008,0.055614,...,0.125347,0.035950,0.131429,0.051656,0.125535,0.044258,0.205714,0.046675,0.169086,0.062659


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>